# 04: Visualize pLLPS-Colored STRING Networks

This notebook creates publication-quality network visualizations for enriched functional groups, with nodes colored by pLLPS score.

**Workflow:**
1. Load interaction networks for each enriched group
2. Build NetworkX graphs with pLLPS node attributes
3. Visualize networks with pLLPS coloring (blue→white→red)
4. Node size represents network degree (connectivity)
5. Edge width represents STRING interaction confidence

**Inputs:**
- `results/string_networks_by_group/{group}_interactions.csv`: Interactions per group
- `results/full_dataset.csv`: Full dataset with pLLPS scores

**Outputs:**
- `results/network_visualizations/{group}_network.png`: High-resolution network plots
- `results/network_statistics_by_group.json`: Network metrics and statistics

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import llps_functions as lf
import json
import importlib
from pathlib import Path
from matplotlib.colors import Normalize, LinearSegmentedColormap

# Reload module
importlib.reload(lf)

# Configure plotting
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 10)
plt.rcParams['font.size'] = 10

# Constants
VIZ_OUTPUT_DIR = Path('results/network_visualizations')
VIZ_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Color map for pLLPS scores (blue=low, red=high)
PLLPS_CMAP = LinearSegmentedColormap.from_list('pllps', ['#0571b0', '#f7f7f7', '#ca0020'])

print("✅ Libraries imported successfully")
print(f"Visualization output directory: {VIZ_OUTPUT_DIR}")

✅ Libraries imported successfully
Visualization output directory: results/network_visualizations


## 1. Load Network Data and Prepare pLLPS Lookup

In [2]:
# Load full dataset for pLLPS scores
df_full = lf.load_analysis_result('full_dataset', format='csv')
pllps_lookup = dict(zip(df_full['Entry'], df_full['p(LLPS)']))

print(f"\n📊 Data Loaded:")
print(f"   pLLPS lookup: {len(pllps_lookup)} proteins")
print(f"   pLLPS range: [{df_full['p(LLPS)'].min():.3f}, {df_full['p(LLPS)'].max():.3f}]")
print(f"   Mean pLLPS: {df_full['p(LLPS)'].mean():.3f}")

# Load summary data
with open('results/enriched_group_interactions_summary.json', 'r') as f:
    summary_data = json.load(f)

groups_to_visualize = [g for g in summary_data.keys() if summary_data[g].get('interaction_count', 0) > 0]
print(f"\n   Groups with interactions: {groups_to_visualize}")

✅ Loaded CSV from: results/full_dataset.csv (20366 rows)

📊 Data Loaded:
   pLLPS lookup: 20366 proteins
   pLLPS range: [0.060, 1.000]
   Mean pLLPS: 0.488

   Groups with interactions: ['Structural', 'Ion Channel', 'Transporter', 'GPCR']


## 2. Build Networks and Calculate Statistics

In [3]:
# Build networks for each group
network_statistics = {}

for group_name in groups_to_visualize:
    print(f"\n{'='*60}")
    print(f"Building network for: {group_name}")
    print(f"{'='*60}")
    
    # Load interactions
    group_file = Path('results/string_networks_by_group') / f"{group_name.lower().replace(' ', '_')}_interactions.csv"
    
    if not group_file.exists():
        print(f"  ⚠️  File not found: {group_file}")
        continue
    
    interactions_df = pd.read_csv(group_file)
    
    # Build NetworkX graph
    G = nx.Graph()
    
    # Add edges with weights
    for _, row in interactions_df.iterrows():
        partner_id = row['partner']
        score = row['combined_score']
        pllps = row.get('partner_pllps', np.nan)
        
        G.add_edge(row['protein'], partner_id, weight=score)
        
        # Set pLLPS attributes
        if pd.notna(pllps):
            G.nodes[partner_id]['pllps'] = pllps
        else:
            G.nodes[partner_id]['pllps'] = pllps_lookup.get(partner_id, np.nan)
        
        # Protein node pLLPS (seed protein)
        if row['protein'] not in G.nodes:
            G.nodes[row['protein']]['pllps'] = pllps_lookup.get(row['protein'], np.nan)
    
    # Calculate statistics
    print(f"\n   Nodes: {G.number_of_nodes()}")
    print(f"   Edges: {G.number_of_edges()}")
    print(f"   Density: {nx.density(G):.4f}")
    
    # Degree statistics
    degrees = [G.degree(n) for n in G.nodes()]
    print(f"   Mean degree: {np.mean(degrees):.2f}")
    print(f"   Max degree: {np.max(degrees)}")
    
    # pLLPS statistics
    pllps_values = [G.nodes[n].get('pllps', np.nan) for n in G.nodes()]
    pllps_values = [p for p in pllps_values if pd.notna(p)]
    print(f"   Mean pLLPS: {np.mean(pllps_values):.3f}")
    print(f"   High pLLPS (>0.7): {sum(p > 0.7 for p in pllps_values)}")
    
    # Store statistics
    network_statistics[group_name] = {
        'nodes': G.number_of_nodes(),
        'edges': G.number_of_edges(),
        'density': float(nx.density(G)),
        'mean_degree': float(np.mean(degrees)),
        'mean_pllps': float(np.mean(pllps_values)),
        'high_pllps_count': int(sum(p > 0.7 for p in pllps_values)),
        'graph_object': G  # Store for visualization
    }

print(f"\n{'='*60}")
print(f"✅ Networks built for {len(network_statistics)} groups")


Building network for: Structural

   Nodes: 342
   Edges: 311
   Density: 0.0053
   Mean degree: 1.82
   Max degree: 8
   Mean pLLPS: 0.535
   High pLLPS (>0.7): 77

Building network for: Ion Channel

   Nodes: 149
   Edges: 282
   Density: 0.0256
   Mean degree: 3.79
   Max degree: 21
   Mean pLLPS: 0.515
   High pLLPS (>0.7): 35

Building network for: Transporter

   Nodes: 22
   Edges: 13
   Density: 0.0563
   Mean degree: 1.18
   Max degree: 5
   Mean pLLPS: 0.265
   High pLLPS (>0.7): 0

Building network for: GPCR

   Nodes: 7
   Edges: 4
   Density: 0.1905
   Mean degree: 1.14
   Max degree: 2
   Mean pLLPS: 0.235
   High pLLPS (>0.7): 0

✅ Networks built for 4 groups


## 3. Visualize Networks with pLLPS Coloring

In [4]:
# Visualize each network
for group_name, stats in network_statistics.items():
    G = stats['graph_object']
    
    if G.number_of_nodes() == 0:
        continue
    
    print(f"\nVisualizing: {group_name}")
    
    # Calculate layout
    pos = nx.spring_layout(G, k=0.5, iterations=50, seed=42)
    
    # Extract node colors and sizes
    node_colors = []
    node_sizes = []
    
    for node in G.nodes():
        pllps = G.nodes[node].get('pllps', 0.5)
        if pd.isna(pllps):
            pllps = 0.5
        node_colors.append(pllps)
        
        # Node size based on degree
        degree = G.degree(node)
        size = 300 + (degree * 100)
        node_sizes.append(size)
    
    # Extract edge widths
    edge_widths = [G[u][v]['weight']/1000.0 for u, v in G.edges()]
    
    # Create figure
    fig, ax = plt.subplots(figsize=(14, 12))
    
    # Draw edges
    nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.3, ax=ax)
    
    # Draw nodes with pLLPS coloring
    nodes = nx.draw_networkx_nodes(
        G, pos,
        node_color=node_colors,
        node_size=node_sizes,
        cmap=PLLPS_CMAP,
        vmin=0, vmax=1,
        ax=ax,
        edgecolors='black',
        linewidths=0.5
    )
    
    # Draw labels (only for high-degree nodes)
    labels = {node: node for node in G.nodes() if G.degree(node) > 2}
    nx.draw_networkx_labels(G, pos, labels, font_size=8, font_weight='bold', ax=ax)
    
    # Add colorbar
    cbar = plt.colorbar(nodes, ax=ax, label='pLLPS Score', fraction=0.046, pad=0.04)
    
    # Title and labels
    ax.set_title(f'{group_name} Protein Interaction Network\n(Node color = pLLPS score, Node size = Degree)', 
                 fontsize=14, fontweight='bold')
    ax.axis('off')
    
    # Save figure
    output_file = VIZ_OUTPUT_DIR / f"{group_name.lower().replace(' ', '_')}_network.png"
    plt.savefig(output_file, dpi=300, bbox_inches='tight')
    print(f"   ✅ Saved: {output_file}")
    plt.close()

# Save network statistics
stats_to_save = {}
for group, stats in network_statistics.items():
    stats_copy = stats.copy()
    stats_copy.pop('graph_object', None)  # Remove graph object before JSON serialization
    stats_to_save[group] = stats_copy

with open('results/network_statistics_by_group.json', 'w') as f:
    json.dump(stats_to_save, f, indent=2)

print(f"\n{'='*60}")
print(f"✅ All networks visualized and saved!")
print(f"\n📊 Output files:")
print(f"   - Network visualizations: results/network_visualizations/")
print(f"   - Statistics summary: results/network_statistics_by_group.json")
print(f"\n🎉 pLLPS-colored STRING networks ready for interpretation!")


Visualizing: Structural
   ✅ Saved: results/network_visualizations/structural_network.png

Visualizing: Ion Channel
   ✅ Saved: results/network_visualizations/ion_channel_network.png

Visualizing: Transporter
   ✅ Saved: results/network_visualizations/transporter_network.png

Visualizing: GPCR
   ✅ Saved: results/network_visualizations/gpcr_network.png

✅ All networks visualized and saved!

📊 Output files:
   - Network visualizations: results/network_visualizations/
   - Statistics summary: results/network_statistics_by_group.json

🎉 pLLPS-colored STRING networks ready for interpretation!
